# Revolving-Credit Default: Feature Engineering

Notebook 01 explored the raw dimensions, daily snapshots, payment and servicing events, and 90-day default outcomes. In this notebook we build and register reusable features **iteratively**:

1. Define the shared **Feature Store Entity** (`CREDIT_ACCOUNT`) and understand how it connects features from different source grains.
2. Introduce each source table once at the step where we need it, inspect sample rows (`.limit(10)`), and test each intermediate transformation before assembling the final feature frame.
3. Combine **Snowpark window features** (point-in-time state, lag deltas, and recency) with **Feature Store tiled aggregations** (`30d`, `90d`, `180d`, and `365d` rolling windows).
4. Spot-check derived values against raw daily snapshots for four sample accounts before registering versioned `FeatureView` objects (`V01`) in `DEV_FEATURE_STORE`.
5. Verify point-in-time feature retrieval (`fs.retrieve_feature_values`) across all registered Feature Views before handing off to Notebook 03 (where feature discovery, feature selection, and immutable `Dataset` generation take place).

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import yaml
from IPython.display import display
from snowflake.ml.feature_store import CreationMode, Entity, Feature, FeatureStore, FeatureView
from snowflake.snowpark import functions as sf
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.window import Window

session = get_active_session()

# Load project.yaml so database, schema, role, warehouse, and drift-cutoff settings
# come from the shared repository contract rather than hardcoded cell values.
config_path = Path("../project.yaml")
if not config_path.is_file():
    config_path = Path("project.yaml")
config = yaml.safe_load(config_path.read_text())

# Define the SQL DIV0NULL function once via sf.function() because it is reused
# across multiple ratio features (returning NULL instead of 0 when the denominator is 0).
div0null = sf.function("DIV0NULL")

feature_version = "V01"

database_name = config["snowflake"]["database"]
raw_schema = config["snowflake"]["schemas"]["raw"]
dev_schema = config["snowflake"]["schemas"]["dev"]
feature_schema = config["snowflake"]["schemas"]["feature_store"]
session.use_role(config["snowflake"]["roles"]["developer"])
session.use_warehouse(config["snowflake"]["warehouse"])
session.use_database(database_name)
session.use_schema(dev_schema)

# Connect to the development Feature Store schema (DEV_FEATURE_STORE).
fs = FeatureStore(
    session=session,
    database=database_name,
    name=feature_schema,
    default_warehouse=config["snowflake"]["warehouse"],
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

## 1. Define the Feature Store Entity (`CREDIT_ACCOUNT`)

In Snowflake Feature Store, an **Entity** represents the business object that features describe (such as a customer, a credit account, or a merchant) and declares the **join key column(s)** used to connect features to prediction rows.

In [ ]:
# Register the CREDIT_ACCOUNT entity keyed by ACCOUNT_ID if it does not already exist
# in DEV_FEATURE_STORE

credit_account = Entity(
            name="CREDIT_ACCOUNT",
            join_keys=["ACCOUNT_ID"],
            desc="Revolving-credit facility entity keyed by ACCOUNT_ID for joining Feature Views.",
        )
        
fs.register_entity(credit_account)
fs.list_entities()

## 2. Develop Account Profile Features (`ACCOUNT_PROFILE_FV`)

Account profile features capture slow-moving customer and facility characteristics at each monthly observation point:
- **Static & tenure attributes:** `PRODUCT_CODE`, `ORIGINATION_CHANNEL`, `ACCOUNT_AGE_MONTHS`, and `CUSTOMER_TENURE_MONTHS` from `DIM_CREDIT_ACCOUNT` and `DIM_CUSTOMER`.
- **Income & credit limit capacity:** `CURRENT_CREDIT_LIMIT`, `CURRENT_INCOME_ESTIMATE`, and `LIMIT_TO_INCOME_RATIO` from month-end `FACT_ACCOUNT_DAILY_SNAPSHOT` and the latest prior `FACT_CUSTOMER_FINANCIAL_SNAPSHOT`.
- **Credit-limit change history:** `MONTHS_SINCE_LIMIT_CHANGE` and `LIMIT_CHANGE_PERCENT_12M` from `FACT_ACCOUNT_EVENT` (`EVENT_TYPE = 'CREDIT_LIMIT_CHANGE'`).

We build and verify these in four small steps: first inspect the source tables (`.limit(10)`), then isolate limit-change events and month-end snapshots, then align the point-in-time joins, and finally compute the derived ratios using `div0null`.

In [ ]:
# Define each source table needed for account profile and daily servicing features once
# as a lazy Snowpark DataFrame, then inspect a 10-row sample to confirm columns and dates.
accounts = session.table(f"{database_name}.{raw_schema}.DIM_CREDIT_ACCOUNT")
customers = session.table(f"{database_name}.{raw_schema}.DIM_CUSTOMER")
financials = session.table(f"{database_name}.{raw_schema}.FACT_CUSTOMER_FINANCIAL_SNAPSHOT")
events = session.table(f"{database_name}.{raw_schema}.FACT_ACCOUNT_EVENT")
daily = session.table(f"{database_name}.{raw_schema}.FACT_ACCOUNT_DAILY_SNAPSHOT")
display("CREDIT ACCOUNTS:")
display(accounts.sort("ACCOUNT_ID").limit(10))
display("CUSTOMERS:")
display(customers.sort("CUSTOMER_ID").limit(10))
display("FINANCIALS:")
display(financials.sort("CUSTOMER_ID", "SNAPSHOT_DATE").limit(10))

In [ ]:
# Filter FACT_ACCOUNT_EVENT to credit-limit changes and cast string event payloads to numeric values.
limit_events = (events
                .filter(sf.col("EVENT_TYPE") == "CREDIT_LIMIT_CHANGE")
                .select("ACCOUNT_ID"
                        , sf.to_date("EVENT_TIMESTAMP").alias("EVENT_DATE")
                        , sf.col("OLD_VALUE").cast("double").alias("OLD_LIMIT")
                        , sf.col("NEW_VALUE").cast("double").alias("NEW_LIMIT")
                    )
                )

# Keep only month-end rows from FACT_ACCOUNT_DAILY_SNAPSHOT (where adding 1 day lands on the 1st of the next month).
monthly_daily = (daily
                .filter(sf.dayofmonth(sf.dateadd("day", sf.lit(1), "SNAPSHOT_DATE")) == 1)
            )
            
display('CREDIT LIMIT CHANGES:')
display(limit_events.sort("EVENT_DATE", "ACCOUNT_ID").limit(10))
display('MONTH END ROWS:')
display(monthly_daily.sort("SNAPSHOT_DATE", "ACCOUNT_ID").limit(10))

In [ ]:
# Join month-end account snapshotswith customer dimensions, the customer's
# 1st-of-month financial snapshot for that month and prior credit-limit changes.
# Note: EVENT_DATE is NULL on the majority of the account-months because only
# few accounts ever experience a CREDIT_LIMIT_CHANGE event, and only on or
# after that event date.
profile_candidates = (monthly_daily
                        .join(accounts, "ACCOUNT_ID")
                        .join(customers, "CUSTOMER_ID")
                        .join(financials, (customers["CUSTOMER_ID"] == financials["CUSTOMER_ID"]) 
                                            & (financials["SNAPSHOT_DATE"] == sf.trunc(monthly_daily["SNAPSHOT_DATE"], sf.lit("MONTH")))
                            )
                        .join(limit_events, (monthly_daily["ACCOUNT_ID"] == limit_events["ACCOUNT_ID"]) 
                                            & (limit_events["EVENT_DATE"] <= monthly_daily["SNAPSHOT_DATE"])
                                ,how="left"
                            )
                        .select(monthly_daily["ACCOUNT_ID"].alias("ACCOUNT_ID")
                                , monthly_daily["SNAPSHOT_DATE"].alias("MNTH_DAY_SNAPSHOT_DATE")
                                , financials["SNAPSHOT_DATE"].alias("FINANCIAL_SNAPSHOT_DATE")
                                , "EVENT_DATE", "PRODUCT_CODE", "ORIGINATION_CHANNEL"
                                , "OPEN_DATE", "CUSTOMER_SINCE_DATE"
                                , monthly_daily["CREDIT_LIMIT"].alias("CREDIT_LIMIT")
                                , "MONTHLY_INCOME_ESTIMATE", "NEW_LIMIT", "OLD_LIMIT",
                            )
                    )

profile_candidates.sort("MNTH_DAY_SNAPSHOT_DATE", "ACCOUNT_ID").limit(10)

In [ ]:
# Keep the most recent financial snapshot and limit-change event effective by each month-end date.
latest_available = (Window
                        .partition_by("ACCOUNT_ID", "MNTH_DAY_SNAPSHOT_DATE")
                        .order_by(sf.col("FINANCIAL_SNAPSHOT_DATE").desc()
                                    , sf.col("EVENT_DATE").desc_nulls_last())
                    )

# Shift MNTH_DAY_SNAPSHOT_DATE forward by +1 day to PROFILE_TS so end-of-month state becomes
# available at midnight on the 1st of the following month without same-day leakage.
# When EVENT_DATE is NULL (no limit change yet), LIMIT_CHANGE_PERCENT_12M defaults to 0.0
# while MONTHS_SINCE_LIMIT_CHANGE stays NULL ("never changed" rather than 0 months ago).
profile_df = (profile_candidates
                .with_column("AVAILABILITY_ORDER"
                            , sf.row_number().over(latest_available)
                )
                .filter(sf.col("AVAILABILITY_ORDER") == 1)
                .select("ACCOUNT_ID"
                        , sf.dateadd("day", sf.lit(1), sf.col("MNTH_DAY_SNAPSHOT_DATE")).cast("timestamp_ntz").alias("PROFILE_TS")
                        , "PRODUCT_CODE", "ORIGINATION_CHANNEL"
                        , sf.datediff("month", sf.col("OPEN_DATE"), sf.col("MNTH_DAY_SNAPSHOT_DATE")).alias("ACCOUNT_AGE_MONTHS")
                        , sf.datediff("month", sf.col("CUSTOMER_SINCE_DATE"), sf.col("MNTH_DAY_SNAPSHOT_DATE")).alias("CUSTOMER_TENURE_MONTHS")
                        , sf.col("CREDIT_LIMIT").alias("CURRENT_CREDIT_LIMIT")
                        , sf.col("MONTHLY_INCOME_ESTIMATE").alias("CURRENT_INCOME_ESTIMATE")
                        , div0null(sf.col("CREDIT_LIMIT"), sf.col("MONTHLY_INCOME_ESTIMATE")).alias("LIMIT_TO_INCOME_RATIO")
                        , sf.datediff("month", sf.col("EVENT_DATE"), sf.col("MNTH_DAY_SNAPSHOT_DATE")).alias("MONTHS_SINCE_LIMIT_CHANGE")
                        , sf.iff(sf.col("EVENT_DATE") >= sf.dateadd("month", sf.lit(-12), sf.col("MNTH_DAY_SNAPSHOT_DATE"))
                                , div0null(sf.col("NEW_LIMIT") - sf.col("OLD_LIMIT"), sf.col("OLD_LIMIT"))
                                ,sf.lit(0.0)
                            ).alias("LIMIT_CHANGE_PERCENT_12M")
                )
            )

profile_df.sort("PROFILE_TS", "ACCOUNT_ID").limit(10)

In [ ]:
profile_view_draft = FeatureView(
    name="ACCOUNT_PROFILE_FV",
    entities=[credit_account],
    feature_df=profile_df,
    timestamp_col="PROFILE_TS",
    refresh_freq="1 day",
    desc="Point-in-time account and customer profile.",
)

# Add descriptions for each feature
profile_view_draft.attach_feature_desc({
        "PRODUCT_CODE": "Credit product family; REVOLVING_CREDIT for this portfolio.",
        "ORIGINATION_CHANNEL": "Channel through which the facility was opened.",
        "ACCOUNT_AGE_MONTHS": "Whole months from facility opening to feature timestamp.",
        "CUSTOMER_TENURE_MONTHS": "Whole months from customer relationship start to feature timestamp.",
        "CURRENT_CREDIT_LIMIT": "Credit limit effective at the feature timestamp, in account currency units.",
        "CURRENT_INCOME_ESTIMATE": "Most recent monthly income estimate available at the feature timestamp.",
        "LIMIT_TO_INCOME_RATIO": "Effective credit limit divided by monthly income estimate.",
        "MONTHS_SINCE_LIMIT_CHANGE": "Whole months since the latest prior credit-limit change; null when none exists.",
        "LIMIT_CHANGE_PERCENT_12M": "Relative size of the latest limit change in the preceding 12 months; zero when none exists.",
})
print(profile_view_draft.status)
profile_view_registered = fs.register_feature_view(profile_view_draft, version=feature_version, overwrite=True)
print(profile_view_registered.status)

The first-day monthly rows provide the point-in-time profile. Null limit-change features mean no prior change, which is a distinct and meaningful state for model preprocessing.

## 3. Develop balance and delinquency features

Daily history supports true calendar windows. Current-state, lag deltas, and days-since stay in Snowpark because they are not rolling aggregations. Trailing utilisation, delinquency-episode, payment, and contact windows are defined later with `Feature` tiles.

In [ ]:
# Order daily snapshots per account chronologically to compute lag transitions and running state.
account_order = Window.partition_by("ACCOUNT_ID").order_by("SNAPSHOT_DATE")

# Enrich daily snapshots with:
# 1. UTILISATION: daily balance-to-limit ratio via div0null
# 2. DELINQUENCY_START: 1.0 when an account transitions from 0 DPD on the prior day to >0 DPD today
# 3. LATEST_DELINQUENT_DATE: most recent date with DAYS_PAST_DUE > 0 up to the current day
servicing = (daily
                .with_column("UTILISATION"
                            , div0null(sf.col("OUTSTANDING_BALANCE"), sf.col("CREDIT_LIMIT")))
                .with_column("DELINQUENCY_START"
                            , sf.iff((sf.col("DAYS_PAST_DUE") > 0) 
                                        & (sf.coalesce(sf.lag("DAYS_PAST_DUE").over(account_order), sf.lit(0)) == 0)
                                    , 1.0, 0.0,)
                            )
                .with_column("LATEST_DELINQUENT_DATE"
                            , sf.max(sf.iff(sf.col("DAYS_PAST_DUE") > 0
                                    , sf.col("SNAPSHOT_DATE"), sf.lit(None))).over(account_order.rows_between(Window.UNBOUNDED_PRECEDING, Window.CURRENT_ROW))
                            )
            )

servicing.sort("SNAPSHOT_DATE", "ACCOUNT_ID").limit(10)

In [ ]:
# Look up the balance exactly 30 and 90 calendar days earlier by date (not by row offset),
# so a missing daily snapshot yields NULL instead of silently comparing against the wrong day.
def balance_days_ago(days):
    """Daily balances keyed by their own (historical) snapshot date, for an N-day lookback join."""
    return daily.select(
        sf.col("ACCOUNT_ID").alias(f"LAG{days}_ACCOUNT_ID"),
        sf.col("SNAPSHOT_DATE").alias(f"LAG{days}_SNAPSHOT_DATE"),
        sf.col("OUTSTANDING_BALANCE").alias(f"BALANCE_{days}D_AGO"),
    )


balance_30d_ago = balance_days_ago(30)
balance_90d_ago = balance_days_ago(90)

# Month-end servicing snapshots; each joins to the balance from SNAPSHOT_DATE - 30 / - 90 days.
month_end = servicing.filter(
    sf.dayofmonth(sf.dateadd("day", sf.lit(1), sf.col("SNAPSHOT_DATE"))) == 1
)

# Derive point-in-time balance state and 30d/90d deltas at month-end, shifting SNAPSHOT_DATE +1 day
# so end-of-month state becomes available on the 1st of the next month (BALANCE_TS).
# Payment-derived ratios are added later to form balance_df; keeping this base frame separate
# makes both cells safe to re-run.
balance_base_df = (month_end
                    .join(balance_30d_ago
                            , (month_end["ACCOUNT_ID"] == balance_30d_ago["LAG30_ACCOUNT_ID"])
                                & (balance_30d_ago["LAG30_SNAPSHOT_DATE"]
                                    == sf.dateadd("day", sf.lit(-30), month_end["SNAPSHOT_DATE"]))
                        , how="left")
                    .join(balance_90d_ago
                            ,(month_end["ACCOUNT_ID"] == balance_90d_ago["LAG90_ACCOUNT_ID"])
                                & (balance_90d_ago["LAG90_SNAPSHOT_DATE"] 
                                    == sf.dateadd("day", sf.lit(-90), month_end["SNAPSHOT_DATE"]))
                            ,how="left")
                    .select("ACCOUNT_ID"
                            , sf.dateadd("day", sf.lit(1), sf.col("SNAPSHOT_DATE")).cast("timestamp_ntz").alias("BALANCE_TS")
                            , sf.col("OUTSTANDING_BALANCE").alias("CURRENT_BALANCE")
                            , sf.col("UTILISATION").alias("CURRENT_UTILISATION")
                            , (sf.col("OUTSTANDING_BALANCE") - sf.col("BALANCE_30D_AGO")).alias("BALANCE_CHANGE_30D")
                            , (sf.col("OUTSTANDING_BALANCE") - sf.col("BALANCE_90D_AGO")).alias("BALANCE_CHANGE_90D")
                            , div0null(sf.col("AVAILABLE_CREDIT"), sf.col("CREDIT_LIMIT")).alias("AVAILABLE_CREDIT_RATIO")
                        )
                )

# Derive point-in-time delinquency state and recency (days since latest delinquent date).
delinquency_df = (servicing
                    .select("ACCOUNT_ID"
                            , sf.dateadd("day", sf.lit(1), sf.col("SNAPSHOT_DATE")).cast("timestamp_ntz").alias("DELINQUENCY_TS")
                            , sf.col("DAYS_PAST_DUE").alias("CURRENT_DAYS_PAST_DUE")
                            , sf.ceil(sf.col("DAYS_PAST_DUE") / sf.lit(30)).alias("CONSECUTIVE_DELINQUENT_MONTHS")
                            , sf.datediff("day", sf.col("LATEST_DELINQUENT_DATE"), sf.col("SNAPSHOT_DATE")).alias("DAYS_SINCE_LAST_DELINQUENCY")
                        )
                    .filter(sf.dayofmonth("DELINQUENCY_TS") == 1)
                )
display('POINT IN TIME BALANCE:')
display(balance_base_df.sort("BALANCE_TS", "ACCOUNT_ID").limit(10))
display('POINT IN TIME DELINQUENCY:')
display(delinquency_df.sort("DELINQUENCY_TS", "ACCOUNT_ID").limit(10))

In [ ]:
delinquency_view_draft = FeatureView(
    name="ACCOUNT_DELINQUENCY_FV",
    entities=[credit_account],
    feature_df=delinquency_df,
    timestamp_col="DELINQUENCY_TS",
    refresh_freq="1 day",
    desc="Current delinquency state and days since last delinquency.",
)

# Add descriptions for each feature
delinquency_view_draft.attach_feature_desc({
        "CURRENT_DAYS_PAST_DUE": "Days past due at the feature timestamp.",
        "CONSECUTIVE_DELINQUENT_MONTHS": "Approximate completed 30-day delinquency periods at the feature timestamp.",
        "DAYS_SINCE_LAST_DELINQUENCY": "Calendar days since the most recent delinquent day; null when none exists.",

})
print(delinquency_view_draft.status)
delinquency_view_registered = fs.register_feature_view(delinquency_view_draft, version=feature_version, overwrite=True)
print(delinquency_view_registered.status)

### Use native tiled aggregation for reusable rolling windows

The daily source supports declarative Feature Store aggregation via the `Feature` class. A one-day tile divides all requested windows (30, 90, 180, 365 days) exactly. Rolling averages, maxima, standard deviation, indicator sums, and delinquency-episode counts live here as tiled aggregations. The Snowpark cell above keeps only current-state, lag deltas, and days-since.

In [ ]:
# Prepare the daily source DataFrame for declarative Feature Store tiled aggregation.
# Shifting SNAPSHOT_DATE +1 day ensures each day's end-of-day balance is eligible from the next day onward.
aggregation_source = (servicing
                        .select("ACCOUNT_ID"
                                , sf.dateadd("day", sf.lit(1), sf.col("SNAPSHOT_DATE")).cast("timestamp_ntz").alias("AGGREGATION_TS")
                                , sf.col("UTILISATION").alias("DAILY_UTILISATION")
                                , sf.col("DAYS_PAST_DUE").cast("double").alias("DAILY_DAYS_PAST_DUE")
                                , sf.iff(sf.col("OUTSTANDING_BALANCE") > sf.col("CREDIT_LIMIT"), sf.lit(1.0), sf.lit(0.0)).alias("IS_OVER_LIMIT")
                                , sf.iff(sf.col("DAYS_PAST_DUE") > 0, sf.lit(1.0), sf.lit(0.0)).alias("IS_DELINQUENT")
                                , sf.col("DELINQUENCY_START").alias("DELINQUENCY_START")
                            )
                    )
aggregation_source.limit(10)

In [ ]:

# Define 30d, 90d, 180d, and 365d rolling aggregations using 1-day Feature Store tiles.
aggregation_features = [
    Feature.avg("DAILY_UTILISATION", "30d").alias("AVG_UTILISATION_30D"),
    Feature.avg("DAILY_UTILISATION", "90d").alias("AVG_UTILISATION_90D"),
    Feature.max("DAILY_UTILISATION", "90d").alias("MAX_UTILISATION_90D"),
    Feature.avg("DAILY_UTILISATION", "365d").alias("AVG_UTILISATION_365D"),
    Feature.max("DAILY_UTILISATION", "365d").alias("MAX_UTILISATION_365D"),
    Feature.stddev("DAILY_UTILISATION", "90d").alias("UTILISATION_STDDEV_90D"),
    Feature.max("DAILY_DAYS_PAST_DUE", "90d").alias("MAX_DAYS_PAST_DUE_3M"),
    Feature.max("DAILY_DAYS_PAST_DUE", "180d").alias("MAX_DAYS_PAST_DUE_6M"),
    Feature.max("DAILY_DAYS_PAST_DUE", "365d").alias("MAX_DAYS_PAST_DUE_365D"),
    Feature.sum("IS_OVER_LIMIT", "30d").alias("DAYS_OVER_LIMIT_30D"),
    Feature.sum("IS_DELINQUENT", "90d").alias("DELINQUENT_DAYS_90D"),
    Feature.sum("DELINQUENCY_START", "180d").alias("DELINQUENCY_EPISODES_6M"),
]

aggregation_view_draft = FeatureView(
    name="ACCOUNT_AGGREGATION_FV",
    entities=[credit_account],
    feature_df=aggregation_source,
    feature_granularity="1 day",
    features=aggregation_features,
    timestamp_col="AGGREGATION_TS",
    refresh_freq="1 day",
    desc="Tiled daily account behaviour aggregated over 30-, 90-, 180-, and 365-day windows.",
)

# Add descriptions for each feature
aggregation_view_draft.attach_feature_desc({
    "AVG_UTILISATION_30D": "Mean daily utilisation over the trailing 30-day window.",
    "AVG_UTILISATION_90D": "Mean daily utilisation over the trailing 90-day window.",
    "MAX_UTILISATION_90D": "Maximum daily utilisation over the trailing 90-day window.",
    "AVG_UTILISATION_365D": "Mean daily utilisation over the trailing 365-day window.",
    "MAX_UTILISATION_365D": "Maximum daily utilisation over the trailing 365-day window.",
    "UTILISATION_STDDEV_90D": "Standard deviation of daily utilisation over the trailing 90-day window.",
    "MAX_DAYS_PAST_DUE_3M": "Maximum days past due in the trailing 90-day window.",
    "MAX_DAYS_PAST_DUE_6M": "Maximum days past due in the trailing 180-day window.",
    "MAX_DAYS_PAST_DUE_365D": "Maximum days past due over the trailing 365-day window.",
    "DAYS_OVER_LIMIT_30D": "Number of days balance exceeded limit in the trailing 30-day window.",
    "DELINQUENT_DAYS_90D": "Days with a positive past-due balance in the trailing 90-day window.",
    "DELINQUENCY_EPISODES_6M": "Transitions from current to delinquent in the trailing 180-day window.",
})
print(aggregation_view_draft.status)
aggregation_view_registered = fs.register_feature_view(aggregation_view_draft, version=feature_version, overwrite=True)
print(aggregation_view_registered.status)

## 4. Develop payment and contact features

Payment and contact events are also tiled with `Feature`. Event-level sources keep the inclusion boundary at the event timestamp; Feature Store then materialises the trailing 30-, 90-, and 180-day windows. Recency, payment-to-balance ratios, and the offset payment trend stay as Snowpark current-state or derived columns because they are not simple rolling aggregations.

In [ ]:
# Define FACT_PAYMENT once and inspect a 10-row sample before building tiled payment features.
payments = session.table(f"{database_name}.{raw_schema}.FACT_PAYMENT")
payments.sort("PAYMENT_DATE", "ACCOUNT_ID").limit(10)

In [ ]:
# Shift PAYMENT_DATE +1 day to PAYMENT_TS so payments posted on day T become available
# at midnight on day T+1, and create a numeric indicator for missed payments.
payment_source = (payments
                    .select("ACCOUNT_ID"
                        , sf.dateadd("day", sf.lit(1), sf.col("PAYMENT_DATE")).cast("timestamp_ntz").alias("PAYMENT_TS")
                        , "PAYMENT_ID"
                        , sf.col("PAYMENT_AMOUNT").cast("double").alias("PAYMENT_AMOUNT")
                        , sf.iff(sf.col("PAYMENT_STATUS") == "MISSED", sf.lit(1.0), sf.lit(0.0)).alias("IS_MISSED")
                        
                    )
                )

payment_source.limit(10)

In [ ]:
# Define 30d, 90d, and 180d rolling aggregations
payment_features = [
    Feature.sum("PAYMENT_AMOUNT", "30d").alias("PAYMENT_SUM_30D"),
    Feature.sum("PAYMENT_AMOUNT", "90d").alias("PAYMENT_SUM_90D"),
    Feature.count("PAYMENT_ID", "90d").alias("PAYMENT_COUNT_90D"),
    Feature.sum("IS_MISSED", "90d").alias("MISSED_PAYMENT_COUNT_3M"),
    Feature.sum("IS_MISSED", "180d").alias("MISSED_PAYMENT_COUNT_6M"),
]

payment_view_draft = FeatureView(
    name="ACCOUNT_PAYMENT_FV",
    entities=[credit_account],
    feature_df=payment_source,
    feature_granularity="1 day",
    features=payment_features,
    timestamp_col="PAYMENT_TS",
    refresh_freq="1 day",
    desc="Tiled payment behaviour from posted and missed payment events.",
)

payment_view_draft.attach_feature_desc({
    "PAYMENT_SUM_30D": "Payment amount posted in the trailing 30-day window.",
    "PAYMENT_SUM_90D": "Payment amount posted in the trailing 90-day window.",
    "PAYMENT_COUNT_90D": "Number of scheduled payment events in the trailing 90-day window.",
    "MISSED_PAYMENT_COUNT_3M": "Missed payment events in the trailing 90-day window.",
    "MISSED_PAYMENT_COUNT_6M": "Missed payment events in the trailing 180-day window.",
})

print(payment_view_draft.status)
payment_view_registered = fs.register_feature_view(payment_view_draft, version=feature_version, overwrite=True)
print(payment_view_registered.status)

In [ ]:
# Define FACT_CUSTOMER_CONTACT once and inspect a 10-row sample before building tiled contact features.
contacts = session.table(f"{database_name}.{raw_schema}.FACT_CUSTOMER_CONTACT")
contacts.sort("CONTACT_TIMESTAMP", "ACCOUNT_ID").limit(10)

In [ ]:
# Shift the contact date +1 day to CONTACT_TS and flag contacts ending in a broken promise to pay.
contact_source = (contacts
                    .select("ACCOUNT_ID"
                            , sf.dateadd("day", sf.lit(1), sf.to_date(sf.col("CONTACT_TIMESTAMP"))).cast("timestamp_ntz").alias("CONTACT_TS")
                            , "CONTACT_ID"
                            , sf.iff(sf.col("CONTACT_OUTCOME") == "BROKEN_PROMISE", sf.lit(1.0), sf.lit(0.0)).alias("IS_BROKEN_PROMISE")
                        )
                )
contact_source.limit(10)

In [ ]:
# Define 30d, and 90d rolling aggregations.
contact_features = [
    Feature.count("CONTACT_ID", "30d").alias("CONTACT_COUNT_30D"),
    Feature.count("CONTACT_ID", "90d").alias("CONTACT_COUNT_90D"),
    Feature.sum("IS_BROKEN_PROMISE", "90d").alias("BROKEN_PROMISE_COUNT_90D"),
]

contact_view_draft = FeatureView(
    name="ACCOUNT_CONTACT_FV",
    entities=[credit_account],
    feature_df=contact_source,
    feature_granularity="1 day",
    features=contact_features,
    timestamp_col="CONTACT_TS",
    refresh_freq="1 day",
    desc="Tiled servicing contact behaviour over 30- and 90-day windows.",
)

contact_view_draft.attach_feature_desc({
    "CONTACT_COUNT_30D": "Servicing contacts in the trailing 30-day window.",
    "CONTACT_COUNT_90D": "Servicing contacts in the trailing 90-day window.",
    "BROKEN_PROMISE_COUNT_90D": "Contacts ending in BROKEN_PROMISE in the trailing 90-day window.",
})

print(contact_view_draft.status)
contact_view_registered = fs.register_feature_view(contact_view_draft, version=feature_version, overwrite=True)
print(contact_view_registered.status)

In [ ]:
# Define ACCOUNT_OBSERVATION once to obtain the monthly feature dates per account.
# Observations have the outcome ie target
observations = session.table(f"{database_name}.{raw_schema}.ACCOUNT_OBSERVATION")

observations.limit(10)

In [ ]:
# Get the distinct date per accounts
observation_dates = (observations
                        .select("ACCOUNT_ID", sf.col("OBSERVATION_DATE").alias("FEATURE_DATE"))
                        .distinct()
                    )

# Capture the prior day's balance and amount due (shifted +1 day to match FEATURE_DATE)
# so 30-day payments can be compared against the balance and amount due at observation time.
prior_servicing = (daily
                    .select("ACCOUNT_ID", sf.dateadd("day", sf.lit(1), sf.col("SNAPSHOT_DATE")).alias("FEATURE_DATE")
                            , sf.col("OUTSTANDING_BALANCE").alias("PRIOR_BALANCE")
                            , sf.col("AMOUNT_DUE").alias("PRIOR_AMOUNT_DUE")
                        )
                )
                
# Join payments strictly before FEATURE_DATE within the trailing 180-day window.
payment_events = (observation_dates
                    .join(payments, (observation_dates["ACCOUNT_ID"] == payments["ACCOUNT_ID"])
                                    & (payments["PAYMENT_DATE"] < observation_dates["FEATURE_DATE"])
                                    & (payments["PAYMENT_DATE"] > sf.dateadd("day", sf.lit(-180), observation_dates["FEATURE_DATE"]))
                        , how="left")
                )

# The join keeps both ACCOUNT_ID columns (Snowpark prefixes them to disambiguate),
# so project unambiguous column names before aggregating.
payment_window = (payment_events
                    .select(observation_dates["ACCOUNT_ID"].alias("ACCOUNT_ID")
                            , observation_dates["FEATURE_DATE"].alias("FEATURE_DATE")
                            , payments["PAYMENT_DATE"].alias("PAYMENT_DATE")
                            , payments["PAYMENT_AMOUNT"].alias("PAYMENT_AMOUNT")
                            , payments["PAYMENT_STATUS"].alias("PAYMENT_STATUS")
                    )
                )
                
# Compute payment-to-balance ratio, payment-to-amount-due ratio, days since last positive payment,
# and the 3-month payment amount trend (recent 30d mean minus days 60..89 mean).
payment_derived = (payment_window
                    .group_by("ACCOUNT_ID", "FEATURE_DATE")
                    .agg(sf.sum(sf.iff(sf.datediff("day", sf.col("PAYMENT_DATE"), sf.col("FEATURE_DATE")) < 30
                                        , sf.col("PAYMENT_AMOUNT"), 0)).alias("_PAYMENT_SUM_30D")
                        , sf.datediff("day", sf.max(sf.iff(sf.col("PAYMENT_AMOUNT") > 0, sf.col("PAYMENT_DATE")
                                                        , sf.lit(None))), sf.col("FEATURE_DATE")).alias("DAYS_SINCE_LAST_PAYMENT")
                        , sf.avg(sf.iff(sf.datediff("day", sf.col("PAYMENT_DATE"), sf.col("FEATURE_DATE")) < 30
                                        , sf.col("PAYMENT_AMOUNT"), sf.lit(None))).alias("AVG_PAYMENT_RECENT_30D")
                        , sf.avg(sf.iff((sf.datediff("day", sf.col("PAYMENT_DATE"), sf.col("FEATURE_DATE")) >= 60) 
                                            & (sf.datediff("day", sf.col("PAYMENT_DATE"), sf.col("FEATURE_DATE")) < 90)
                                        , sf.col("PAYMENT_AMOUNT"), sf.lit(None))).alias("AVG_PAYMENT_EARLY_30D")
                    )
                    .join(prior_servicing, ["ACCOUNT_ID", "FEATURE_DATE"], how="left")
                    .select("ACCOUNT_ID", sf.col("FEATURE_DATE").cast("timestamp_ntz").alias("BALANCE_TS")
                            , div0null(sf.col("_PAYMENT_SUM_30D"), sf.col("PRIOR_BALANCE")).alias("PAYMENT_TO_BALANCE_RATIO_30D")
                            , div0null(sf.col("_PAYMENT_SUM_30D"), sf.col("PRIOR_AMOUNT_DUE")).alias("PAYMENT_TO_AMOUNT_DUE_RATIO_30D")
                            , "DAYS_SINCE_LAST_PAYMENT"
                            , (sf.col("AVG_PAYMENT_RECENT_30D") - sf.col("AVG_PAYMENT_EARLY_30D")).alias("PAYMENT_AMOUNT_TREND_3M")
                    )
                )

balance_df = balance_base_df.join(payment_derived, ["ACCOUNT_ID", "BALANCE_TS"], how="left")
display(balance_df.sort("BALANCE_TS", "ACCOUNT_ID").limit(10))

In [ ]:
balance_view_draft = FeatureView(
    name="ACCOUNT_BALANCE_FV",
    entities=[credit_account],
    feature_df=balance_df,
    timestamp_col="BALANCE_TS",
    refresh_freq="1 day",
    desc="Current balance, lag deltas, recency, and derived payment ratios.",
)

balance_view_draft.attach_feature_desc({
        "CURRENT_BALANCE": "Outstanding balance at the feature timestamp, in account currency units.",
        "CURRENT_UTILISATION": "Outstanding balance divided by effective credit limit.",
        "BALANCE_CHANGE_30D": "Current balance less the balance 30 calendar days earlier.",
        "BALANCE_CHANGE_90D": "Current balance less the balance 90 calendar days earlier.",
        "AVAILABLE_CREDIT_RATIO": "Available credit divided by effective credit limit.",
        "PAYMENT_TO_BALANCE_RATIO_30D": "Trailing 30-day payments divided by the balance available before observation.",
        "PAYMENT_TO_AMOUNT_DUE_RATIO_30D": "Trailing 30-day payments divided by the amount due available before observation.",
        "DAYS_SINCE_LAST_PAYMENT": "Calendar days since the latest positive payment; null when none exists.",
        "PAYMENT_AMOUNT_TREND_3M": "Mean payment in the most recent 30 days less the mean from days 60 through 89.",
})

print(balance_view_draft.status)
balance_view_registered = fs.register_feature_view(balance_view_draft, version=feature_version, overwrite=True)
print(balance_view_registered.status)

## 5. Verify Point-in-Time Retrieval from the Registered Feature Views

Now that all six `V01` Feature Views are registered in `DEV_FEATURE_STORE`, we verify that `fs.retrieve_feature_values()` joins them point-in-time for the 4 sample accounts (including the native 365-day tiled columns).

Feature engineering stops here: the Feature Store now holds a reusable catalog of versioned Feature Views. In **Notebook 03 (`03_credit_default_model_experiment.ipynb`)**, the model developer discovers the available Feature Views in `DEV_FEATURE_STORE`, selects which features to use, defines the embargoed ground-truth observation spines, and generates the immutable training and validation `Dataset` objects.

In [ ]:
# Retrieve point-in-time feature values across all 6 registered Feature Views for the 4 sample accounts,
# including the 365-day tiled columns materialized by the Feature Store.

feature_views = [aggregation_view_registered, payment_view_registered
                , contact_view_registered, profile_view_registered
                , balance_view_registered, delinquency_view_registered]

selected_feature_spine = (observation_dates
                            .join(selected_accounts, "ACCOUNT_ID")
                            .select("ACCOUNT_ID", sf.col("FEATURE_DATE").cast("timestamp_ntz").alias("OBSERVATION_TS"))
                        )
                        
selected_feature_values = fs.retrieve_feature_values(spine_df=selected_feature_spine
                                                    , features=feature_views
                                                    , spine_timestamp_col="OBSERVATION_TS"
                                                    , include_feature_view_timestamp_col=True
                                                    , join_method="cte",
                                                )

(selected_feature_values
    .select("ACCOUNT_ID", "OBSERVATION_TS", "CURRENT_CREDIT_LIMIT"
            , "CURRENT_INCOME_ESTIMATE", "CONTACT_COUNT_90D"
            , "PAYMENT_SUM_30D", "AVG_UTILISATION_365D"
            , "MAX_UTILISATION_365D", "MAX_DAYS_PAST_DUE_365D"
        )
    .sort("ACCOUNT_ID", "OBSERVATION_TS")).limit(10)
